In [1]:
import os
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use('Agg')

output_dir = Path(os.environ['OUTPUT_DATA_DIR'])
df_agg = pd.read_csv(output_dir / 'fmri_stats.tsv', sep='\t')
df_sub = pd.read_csv(output_dir / 'fmri_stats_per_subject.tsv', sep='\t')

In [2]:
# Figure 1: total hours of fMRI per dataset (+ "all" total)
per_ds = df_agg[df_agg['dataset'] != 'all']
all_row = df_agg[df_agg['dataset'] == 'all'].iloc[0]
datasets = per_ds['dataset'].tolist()
values = per_ds['total_duration_h'].tolist() + [all_row['total_duration_h']]
x_labels = datasets + ['all']
colors = ['#1f77b4'] * len(datasets) + ['#ff7f0e']

fig, ax = plt.subplots(figsize=(10, 5), constrained_layout=True)
ax.bar(range(len(x_labels)), values, color=colors)
ax.axvline(len(datasets) - 0.5, color='gray', linewidth=0.8, linestyle='--')
ax.set_xticks(range(len(x_labels)))
ax.set_xticklabels(x_labels, rotation=45, ha='right')
ax.set_ylabel('Hours')
ax.set_title('CNeuroMod: total fMRI acquisition time per dataset', fontweight='bold')

out_path = output_dir / 'figure_fmri_total_hours.png'
fig.savefig(out_path, dpi=150)
plt.show()
print(f'Figure saved to {out_path}')

Figure saved to /home/pbellec/git/cneuromod.all/analysis/cneuromod.all.statistics/output_data/figure_fmri_total_hours.png


/tmp/claude-1000/ipykernel_366021/3758111648.py:19: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [3]:
# Figure 2: min / avg / max of total_duration_h across subjects with data, per dataset
# Only include subjects who have total_runs > 0 (i.e. have data in the dataset)
# Exclude the "all" pseudo-dataset from per-dataset breakdown; compute "all" from its rows separately
df_sub_ds = df_sub[df_sub['dataset'] != 'all']
has_data = df_sub_ds[df_sub_ds['total_runs'] > 0].dropna(subset=['total_duration_h'])

grouped = has_data.groupby('dataset')['total_duration_h']
stats = grouped.agg(['min', 'mean', 'max']).reset_index()
stats.columns = ['dataset', 'min', 'mean', 'max']

# Append "all" row from per-subject "all" rows
all_has_data = df_sub[df_sub['dataset'] == 'all'][df_sub[df_sub['dataset'] == 'all']['total_runs'] > 0].dropna(subset=['total_duration_h'])
all_stats = pd.DataFrame([{
    'dataset': 'all',
    'min': all_has_data['total_duration_h'].min(),
    'mean': all_has_data['total_duration_h'].mean(),
    'max': all_has_data['total_duration_h'].max(),
}])
stats = pd.concat([stats, all_stats], ignore_index=True)

datasets = stats['dataset'].tolist()
n = len(datasets)
x = np.arange(n)
width = 0.25
colors_min  = ['#1f77b4'] * (n - 1) + ['#ff7f0e']
colors_mean = ['#aec7e8'] * (n - 1) + ['#ffbb78']
colors_max  = ['#2ca02c'] * (n - 1) + ['#d62728']

fig, ax = plt.subplots(figsize=(12, 5), constrained_layout=True)
ax.bar(x - width, stats['min'],  width, label='Min',  color=colors_min)
ax.bar(x,          stats['mean'], width, label='Avg',  color=colors_mean)
ax.bar(x + width, stats['max'],  width, label='Max',  color=colors_max)
ax.axvline(n - 1.5, color='gray', linewidth=0.8, linestyle='--')

ax.set_xticks(x)
ax.set_xticklabels(datasets, rotation=45, ha='right')
ax.set_ylabel('Hours')
ax.set_title('CNeuroMod: total fMRI hours per dataset — min / avg / max across subjects with data',
             fontweight='bold')
ax.legend()

out_path = output_dir / 'figure_fmri_hours_min_avg_max.png'
fig.savefig(out_path, dpi=150)
plt.show()
print(f'Figure saved to {out_path}')

Figure saved to /home/pbellec/git/cneuromod.all/analysis/cneuromod.all.statistics/output_data/figure_fmri_hours_min_avg_max.png


/tmp/claude-1000/ipykernel_366021/68299550.py:44: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [4]:
# Figure 3: total acquisition time per dataset per subject (one panel per subject)
subjects = sorted(df_sub['subject'].unique())
datasets = df_agg[df_agg['dataset'] != 'all']['dataset'].tolist()
total_label = 'TOTAL'
x_labels = datasets + [total_label]
n_bars = len(x_labels)

fig, axes = plt.subplots(2, 3, figsize=(18, 9), constrained_layout=True, sharey=True)
axes_flat = axes.flatten()

for ax, subject in zip(axes_flat, subjects):
    sub_df = df_sub[(df_sub['subject'] == subject) & (df_sub['dataset'] != 'all')].set_index('dataset')
    all_row = df_sub[(df_sub['subject'] == subject) & (df_sub['dataset'] == 'all')]
    values = []
    for ds in datasets:
        if ds in sub_df.index and sub_df.loc[ds, 'total_runs'] > 0:
            values.append(sub_df.loc[ds, 'total_duration_h'])
        else:
            values.append(0.0)
    total = all_row['total_duration_h'].values[0] if len(all_row) > 0 else sum(v for v in values if v > 0)
    values_with_total = values + [total]

    colors = ['#1f77b4'] * len(datasets) + ['#ff7f0e']
    ax.bar(range(n_bars), values_with_total, color=colors)
    ax.yaxis.grid(True, linestyle='--', linewidth=0.5, alpha=0.7)
    ax.set_axisbelow(True)
    ax.set_xticks(range(n_bars))
    ax.set_xticklabels(x_labels, rotation=45, ha='right', fontsize=8)
    ax.set_title(subject, fontweight='bold')
    ax.set_ylabel('Hours')

fig.suptitle('CNeuroMod: total fMRI acquisition time per dataset per subject', fontweight='bold', fontsize=13)

out_path = output_dir / 'total_acquisition_time_per_dataset_per_subject.png'
fig.savefig(out_path, dpi=150)
plt.show()
print(f'Figure saved to {out_path}')

Figure saved to /home/pbellec/git/cneuromod.all/analysis/cneuromod.all.statistics/output_data/total_acquisition_time_per_dataset_per_subject.png


/tmp/claude-1000/ipykernel_366021/858622862.py:36: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [5]:
# Figure 4: total volumes per dataset (+ "all" total)
per_ds = df_agg[df_agg['dataset'] != 'all']
all_row = df_agg[df_agg['dataset'] == 'all'].iloc[0]
datasets = per_ds['dataset'].tolist()
values = per_ds['total_volumes'].tolist() + [all_row['total_volumes']]
x_labels = datasets + ['all']
colors = ['#1f77b4'] * len(datasets) + ['#ff7f0e']

fig, ax = plt.subplots(figsize=(10, 5), constrained_layout=True)
ax.bar(range(len(x_labels)), values, color=colors)
ax.axvline(len(datasets) - 0.5, color='gray', linewidth=0.8, linestyle='--')
ax.set_xticks(range(len(x_labels)))
ax.set_xticklabels(x_labels, rotation=45, ha='right')
ax.set_ylabel('Volumes')
ax.set_title('CNeuroMod: total fMRI volumes per dataset', fontweight='bold')

out_path = output_dir / 'figure_fmri_total_volumes.png'
fig.savefig(out_path, dpi=150)
plt.show()
print(f'Figure saved to {out_path}')

Figure saved to /home/pbellec/git/cneuromod.all/analysis/cneuromod.all.statistics/output_data/figure_fmri_total_volumes.png


/tmp/claude-1000/ipykernel_366021/2873445097.py:19: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [6]:
# Figure 5: min / avg / max of total_volumes across subjects with data, per dataset
df_sub_ds = df_sub[df_sub['dataset'] != 'all']
has_data = df_sub_ds[df_sub_ds['total_runs'] > 0].dropna(subset=['total_volumes'])

grouped = has_data.groupby('dataset')['total_volumes']
stats = grouped.agg(['min', 'mean', 'max']).reset_index()
stats.columns = ['dataset', 'min', 'mean', 'max']

# Append "all" row from per-subject "all" rows
all_has_data = df_sub[df_sub['dataset'] == 'all'][df_sub[df_sub['dataset'] == 'all']['total_runs'] > 0].dropna(subset=['total_volumes'])
all_stats = pd.DataFrame([{
    'dataset': 'all',
    'min': all_has_data['total_volumes'].min(),
    'mean': all_has_data['total_volumes'].mean(),
    'max': all_has_data['total_volumes'].max(),
}])
stats = pd.concat([stats, all_stats], ignore_index=True)

datasets = stats['dataset'].tolist()
n = len(datasets)
x = np.arange(n)
width = 0.25
colors_min  = ['#1f77b4'] * (n - 1) + ['#ff7f0e']
colors_mean = ['#aec7e8'] * (n - 1) + ['#ffbb78']
colors_max  = ['#2ca02c'] * (n - 1) + ['#d62728']

fig, ax = plt.subplots(figsize=(12, 5), constrained_layout=True)
ax.bar(x - width, stats['min'],  width, label='Min',  color=colors_min)
ax.bar(x,          stats['mean'], width, label='Avg',  color=colors_mean)
ax.bar(x + width, stats['max'],  width, label='Max',  color=colors_max)
ax.axvline(n - 1.5, color='gray', linewidth=0.8, linestyle='--')

ax.set_xticks(x)
ax.set_xticklabels(datasets, rotation=45, ha='right')
ax.set_ylabel('Volumes')
ax.set_title('CNeuroMod: total fMRI volumes per dataset — min / avg / max across subjects with data',
             fontweight='bold')
ax.legend()

out_path = output_dir / 'figure_fmri_volumes_min_avg_max.png'
fig.savefig(out_path, dpi=150)
plt.show()
print(f'Figure saved to {out_path}')

Figure saved to /home/pbellec/git/cneuromod.all/analysis/cneuromod.all.statistics/output_data/figure_fmri_volumes_min_avg_max.png


/tmp/claude-1000/ipykernel_366021/811665770.py:42: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [7]:
# Figure 6: total volumes per dataset per subject (one panel per subject)
subjects = sorted(df_sub['subject'].unique())
datasets = df_agg[df_agg['dataset'] != 'all']['dataset'].tolist()
total_label = 'TOTAL'
x_labels = datasets + [total_label]
n_bars = len(x_labels)

fig, axes = plt.subplots(2, 3, figsize=(18, 9), constrained_layout=True, sharey=True)
axes_flat = axes.flatten()

for ax, subject in zip(axes_flat, subjects):
    sub_df = df_sub[(df_sub['subject'] == subject) & (df_sub['dataset'] != 'all')].set_index('dataset')
    all_row = df_sub[(df_sub['subject'] == subject) & (df_sub['dataset'] == 'all')]
    values = []
    for ds in datasets:
        if ds in sub_df.index and sub_df.loc[ds, 'total_runs'] > 0:
            v = sub_df.loc[ds, 'total_volumes']
            values.append(float(v) if v == v else 0.0)
        else:
            values.append(0.0)
    total = float(all_row['total_volumes'].values[0]) if len(all_row) > 0 else sum(v for v in values if v > 0)
    values_with_total = values + [total]

    colors = ['#1f77b4'] * len(datasets) + ['#ff7f0e']
    ax.bar(range(n_bars), values_with_total, color=colors)
    ax.yaxis.grid(True, linestyle='--', linewidth=0.5, alpha=0.7)
    ax.set_axisbelow(True)
    ax.set_xticks(range(n_bars))
    ax.set_xticklabels(x_labels, rotation=45, ha='right', fontsize=8)
    ax.set_title(subject, fontweight='bold')
    ax.set_ylabel('Volumes')

fig.suptitle('CNeuroMod: total fMRI volumes per dataset per subject', fontweight='bold', fontsize=13)

out_path = output_dir / 'total_volumes_per_dataset_per_subject.png'
fig.savefig(out_path, dpi=150)
plt.show()
print(f'Figure saved to {out_path}')

Figure saved to /home/pbellec/git/cneuromod.all/analysis/cneuromod.all.statistics/output_data/total_volumes_per_dataset_per_subject.png


/tmp/claude-1000/ipykernel_366021/1188706309.py:37: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
